# U05_C3 — Reading BLAST Results
**Course:** Concepts of Bioinformatics (CBBI-501) · **Unit 5:** Sequence Alignment and Database Searching
**Type:** Course notebook · **Time:** 40 min · **Difficulty:** ★★☆
**Goes with:** Lectures L14–L15, Practical S6 (BLAST)
**Before this:** U05_C2

## 1. What you will learn
- **Explain** why heuristic search algorithms (BLAST and FASTA) are required for querying massive sequence databases [LO4].
- **Describe** the core algorithmic phases of BLAST: word matching ($W$), neighborhood threshold ($T$), and extension into High-Scoring Segment Pairs (HSPs) [LO4].
- **Differentiate** raw score, bit score ($S'$), and Expectation value ($E$-value) [LO4].
- **Parse** and filter tabular BLAST results (`-outfmt 6`) to detect true homologues and identify low-complexity artifacts [LO4].


## 2. Biological question
The NCBI RefSeq database contains hundreds of millions of protein sequences. Searching this entire database with exact Smith–Waterman dynamic programming would take hours per query. BLAST (Basic Local Alignment Search Tool) finds hits in seconds by indexing short 'words' and extending seeds. When BLAST returns a table of hits, how do we distinguish genuine evolutionary orthologues from chance statistical matches or repetitive sequence artifacts?


In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

NOTEBOOK_ID = "U05_C3"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
print("Python", sys.version.split()[0], "| Biopython", Bio.__version__)

## 3. Concept 1: The BLAST Algorithmic Strategy
### Theory
BLAST replaces the full $O(M \times N)$ matrix search with an indexed heuristic:
1. **Seeding**: Break query into words of length $W$ (default $W=3$ for blastp, $W=11$ for blastn).
2. **Neighborhood Evaluation**: For each query word, list all possible words whose substitution score with the query word is $\ge T$ (threshold).
3. **Database Scanning**: Rapidly scan the database index for exact matches to the neighborhood words.
4. **Extension**: Extend matches in both directions without gaps, then with dynamic programming, forming **High-Scoring Segment Pairs (HSPs)** until score falls below threshold $X$.


In [ ]:
# Worked Example: Generating 3-letter words from HBB peptide
hbb_peptide = "MVHLTPEEKSAV"
W = 3
words = [hbb_peptide[i:i+W] for i in range(len(hbb_peptide) - W + 1)]

print(f"Peptide: {hbb_peptide}")
print(f"Generated {len(words)} query words of length W={W}:")
print(words)


## 4. Concept 2: Bit Score and E-Value
### Theory
The raw alignment score $S$ depends on the scoring matrix and gap penalties used. To compare alignments across different search setups:
1. **Bit Score ($S'$)**: Normalized score independent of matrix units and database size:
   $$S' = \frac{\lambda S - \ln K}{\ln 2}$$
   Higher bit score always means a better alignment.
2. **Expectation Value ($E$-value)**: The number of distinct alignments with score $\ge S$ expected to occur purely by chance in a database of size $N$ with query length $m$:
   $$E = K \cdot m \cdot N \cdot e^{-\lambda S} = m \cdot N \cdot 2^{-S'}$$
- $E \le 10^{-5}$: Strong statistical evidence of biological homology.
- If database size $N$ doubles, $E$-value **doubles** for the exact same alignment!


In [ ]:
# Worked Example: Computing E-value relationship
import math

# Suppose query length m = 147 aa, database size N = 10^8 residues, bit score S_prime = 50
m = 147
N = 10**8
S_prime = 50

# E = m * N * 2^(-S_prime)
e_val = m * N * (2 ** (-S_prime))
print(f"Query length: {m} aa, DB size: {N:.1e} residues")
print(f"Bit score: {S_prime} bits -> Expected random hits (E-value): {e_val:.3e}")

# What if database size increases 10-fold?
e_val_large_db = m * (10 * N) * (2 ** (-S_prime))
print(f"Same alignment in 10x larger database -> E-value: {e_val_large_db:.3e}")


## 5. Concept 3: Reading Tabular BLAST Output (-outfmt 6)
### Theory
Standard tabular BLAST output (`-outfmt 6`) contains 12 tab-separated columns:
1. `qseqid`: Query Seq-id
2. `sseqid`: Subject (database hit) Seq-id
3. `pident`: Percentage of identical matches
4. `length`: Alignment length
5. `mismatch`: Number of mismatches
6. `gapopen`: Number of gap openings
7. `qstart`: Start of alignment in query
8. `qend`: End of alignment in query
9. `sstart`: Start of alignment in subject
10. `send`: End of alignment in subject
11. `evalue`: Expect value
12. `bitscore`: Bit score


In [ ]:
# Worked Example: Loading and filtering saved BLAST results for HBB
import pandas as pd

blast_file = DATA_DIR / "expected" / "blast_hbb_vs_refseq.tsv"
df_blast = pd.read_csv(blast_file, sep="\t")

print("=== Tabular BLAST Results for Human HBB ===")
print(df_blast[["subject_id", "identity", "evalue", "bit_score", "subject_title"]])


In [ ]:
# Try it: Filter hits with evalue < 1e-10 and print their titles
significant_hits = df_blast[df_blast["evalue"] < 1e-10]
print(f"\nNumber of statistically significant hits (E < 1e-10): {len(significant_hits)}")
for idx, row in significant_hits.iterrows():
    print(f"  {row['subject_id']:12} | {row['identity']:6.1f}% id | E={row['evalue']} | {row['subject_title']}")


## 6. Summary
- BLAST uses word seeds ($W$) and neighborhood scoring thresholds ($T$) to accelerate searching by orders of magnitude over full dynamic programming.
- Bit score $S'$ is matrix-independent; $E$-value scales proportionally with database size $N$.
- Tabular format (`-outfmt 6`) allows filtering by percent identity, coverage, and E-value in Pandas.


## 7. Next Steps
- Complete alignment exercises in `U05_P_Sequence_Alignment.ipynb`.
- Test your sequence alignment and BLAST knowledge in `U05_Q_Sequence_Alignment.ipynb`.
